# Step 1: Extract Audio & Text Embeddings with CLAP

In [2]:
import os
import numpy as np
import pandas as pd
import laion_clap

PROJECT_DIR = os.path.dirname(os.getcwd())
AUDIO_DIR   = os.path.join(PROJECT_DIR, 'data', 'audio')
CSV_PATH    = os.path.join(PROJECT_DIR, 'data', 'metadata_500.csv')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
EMBED_DIR = os.path.join(RESULTS_DIR, 'embeddings')
os.makedirs(EMBED_DIR, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('CSV exists:', os.path.exists(CSV_PATH))
print('Audio dir exists:', os.path.exists(AUDIO_DIR))

/opt/anaconda3/envs/dl4m_audioText/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_DIR: /Users/liann77/One/NYUStudy/NYU26Spring/DL4M/finalGroupWorkClap
CSV exists: True
Audio dir exists: True


In [3]:
df = pd.read_csv(CSV_PATH)

audio_paths = [os.path.join(AUDIO_DIR, row['audio_filename']) for _, row in df.iterrows()]
captions    = df['caption'].tolist()

print(f'total number of samples : {len(audio_paths)}')
print('path of first sample:', audio_paths[0])
print('caption of first sample:', captions[0])
print('sample exists!:', os.path.exists(audio_paths[0]))

total number of samples : 500
path of first sample: /Users/liann77/One/NYUStudy/NYU26Spring/DL4M/finalGroupWorkClap/data/audio/0.wav
caption of first sample: constant rattling noise and sharp vibrations
sample exists!: True


In [4]:
import soundfile as sf

bad_files = []
for path in audio_paths:
    try:
        data, sr = sf.read(path)
        if len(data) == 0:
            bad_files.append((path, 'empty'))
    except Exception as e:
        bad_files.append((path, str(e)))

print(f'total files: {len(audio_paths)}')
print(f'corrupted files: {len(bad_files)}')
if bad_files:
    for p, reason in bad_files:
        print(f'  BAD: {os.path.basename(p)} — {reason}')
else:
    print('all files OK, proceeding to extract embeddings')

total files: 500
corrupted files: 3
  BAD: 34.wav — empty
  BAD: 196.wav — empty
  BAD: 247.wav — empty


In [5]:
bad_set = set(p for p, _ in bad_files)

clean_audio_paths = [p for p in audio_paths if p not in bad_set]
clean_captions    = [c for p, c in zip(audio_paths, captions) if p not in bad_set]
clean_df          = df[[p not in bad_set for p in audio_paths]].reset_index(drop=True)

print(f'samples after filtering: {len(clean_audio_paths)}')

samples after filtering: 497


In [6]:
# first run auto-downloads pretrained weights (~600MB)
model = laion_clap.CLAP_Module(enable_fusion=False)
model.load_ckpt()
print('model loaded successfully!')

/opt/anaconda3/envs/dl4m_audioText/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1729647065806/work/aten/src/ATen/native/TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 17085.98it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not o

Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.we

In [7]:
print('extracting audio embeddings, this may take a few minutes...')
audio_embeddings = model.get_audio_embedding_from_filelist(x=clean_audio_paths, use_tensor=False)
print('audio embeddings shape:', audio_embeddings.shape)

extracting audio embeddings, this may take a few minutes...
audio embeddings shape: (497, 512)


In [8]:
print('extracting text embeddings...')
text_embeddings = model.get_text_embedding(clean_captions, use_tensor=False)
print('text embeddings shape:', text_embeddings.shape)

extracting text embeddings...
text embeddings shape: (497, 512)


In [9]:
np.save(os.path.join(EMBED_DIR, 'audio_embeddings.npy'), audio_embeddings)
np.save(os.path.join(EMBED_DIR, 'text_embeddings.npy'),  text_embeddings)
print('saved!')
print('audio_embeddings.npy ->', audio_embeddings.shape)
print('text_embeddings.npy  ->', text_embeddings.shape)

saved!
audio_embeddings.npy -> (497, 512)
text_embeddings.npy  -> (497, 512)
